### Outliers ville précise

In [2]:
import pandas as pd

# Configurer pandas pour afficher les colonnes longues en entier
pd.set_option('display.max_colwidth', None)

# Demander à l'utilisateur la ville à chercher
ville = input("Entrez le nom de la ville à chercher: ")

# Charger tous les CSV STEP02
files = [
    '../csv/STEP02/STEP02_maisons_dept22.csv',
    '../csv/STEP02/STEP02_maisons_dept29.csv',
    '../csv/STEP02/STEP02_maisons_dept35.csv',
    '../csv/STEP02/STEP02_maisons_dept56.csv'
]

dfs = [pd.read_csv(f, encoding='utf-8') for f in files]
df = pd.concat(dfs, ignore_index=True)

# Filtrer les maisons pour la ville donnée
maisons_ville = df[df['Lieu'].str.contains(ville, na=False)].copy()

print(f"Maisons à {ville} ({len(maisons_ville)} au total):")
if len(maisons_ville) > 0:
    # Limiter l'affichage pour le tableau principal
    pd.set_option('display.max_colwidth', 50)
    display(maisons_ville.sort_values('Prix au m2'))
    # Remettre à None pour les prints suivants
    pd.set_option('display.max_colwidth', None)

    # Nettoyage pour calculs
    maisons_ville['Prix au m2'] = pd.to_numeric(maisons_ville['Prix au m2'], errors='coerce')

    # Moyenne avec outliers
    mean_ville = maisons_ville['Prix au m2'].mean()
    print(f"\nMoyenne {ville}: {mean_ville:.2f} €/m²")

    # --- Détection outliers avec IQR ---
    Q1_ville = maisons_ville['Prix au m2'].quantile(0.25)
    Q3_ville = maisons_ville['Prix au m2'].quantile(0.75)
    IQR_ville = Q3_ville - Q1_ville

    print(f"IQR - Q1: {Q1_ville:.2f}, Q3: {Q3_ville:.2f}, IQR: {IQR_ville:.2f}")

    lower_ville = Q1_ville - 1.5 * IQR_ville
    upper_ville = Q3_ville + 1.5 * IQR_ville

    outliers_ville = maisons_ville[(maisons_ville['Prix au m2'] < lower_ville) | (maisons_ville['Prix au m2'] > upper_ville)]
    print(f"Outliers selon IQR ({len(outliers_ville)}):")
    print(outliers_ville[['Prix au m2', 'Prix', 'Taille', 'Lien']].sort_values('Prix au m2'))

    # Moyenne sans outliers
    df_sans_outliers = maisons_ville[~maisons_ville.index.isin(outliers_ville.index)]
    mean_sans_outliers = df_sans_outliers['Prix au m2'].mean()
    print(f"Moyenne sans outliers: {mean_sans_outliers:.2f} €/m²")

else:
    print("Aucune maison trouvée dans le dataset pour cette ville.")

FileNotFoundError: [Errno 2] No such file or directory: '../csv/STEP02/STEP02_maisons_dept22.csv'

### Outliers département

In [1]:
# Calculer les outliers par ville pour un département donné
import pandas as pd

# Configurer pandas pour limiter l'affichage des colonnes longues
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', None)

# Demander le département
dept = input("Entrez le numéro du département (22, 29, 35 ou 56): ")

# Charger le CSV correspondant
file_path = f'../csv/STEP02/STEP02_maisons_dept{dept}.csv'
df = pd.read_csv(file_path, encoding='utf-8')

# Obtenir les villes uniques
villes = df['Lieu'].unique()

# Liste pour stocker les résultats
resultats = []
# Liste pour stocker les dataframes nettoyés
dfs_clean = []

for ville in villes:
    maisons_ville = df[df['Lieu'] == ville].copy()
    
    if len(maisons_ville) > 0:
        # Nettoyage pour calculs
        maisons_ville['Prix au m2'] = pd.to_numeric(maisons_ville['Prix au m2'], errors='coerce')
        maisons_ville = maisons_ville.dropna(subset=['Prix au m2'])
        
        nombre_maisons = len(maisons_ville)
        moyenne_avant = maisons_ville['Prix au m2'].mean()
        
        if len(maisons_ville) >= 4:  # Au moins 4 points pour calculer IQR
            # --- Détection outliers avec IQR ---
            Q1_ville = maisons_ville['Prix au m2'].quantile(0.25)
            Q3_ville = maisons_ville['Prix au m2'].quantile(0.75)
            IQR_ville = Q3_ville - Q1_ville

            lower_ville = Q1_ville - 1.5 * IQR_ville
            upper_ville = Q3_ville + 1.5 * IQR_ville

            outliers_ville = maisons_ville[(maisons_ville['Prix au m2'] < lower_ville) | (maisons_ville['Prix au m2'] > upper_ville)]
            nombre_outliers = len(outliers_ville)
            
            # Moyenne après outliers
            df_sans_outliers = maisons_ville[~maisons_ville.index.isin(outliers_ville.index)]
            moyenne_apres = df_sans_outliers['Prix au m2'].mean() if len(df_sans_outliers) > 0 else 0
        else:
            nombre_outliers = 0
            moyenne_apres = moyenne_avant  # Pas d'outliers retirés
            df_sans_outliers = maisons_ville  # Garder tout
    else:
        nombre_maisons = 0
        nombre_outliers = 0
        moyenne_avant = 0
        moyenne_apres = 0
        df_sans_outliers = maisons_ville  # Vide
    
    resultats.append({
        'Ville': ville, 
        'Nombre de maisons': nombre_maisons, 
        'Nombre d\'outliers': nombre_outliers,
        'Moyenne avant outliers (€/m²)': round(moyenne_avant, 2),
        'Moyenne après outliers (€/m²)': round(moyenne_apres, 2)
    })
    
    # Ajouter le dataframe nettoyé
    dfs_clean.append(df_sans_outliers)

# Créer un DataFrame avec les résultats
df_outliers = pd.DataFrame(resultats)

# Trier par nombre d'outliers décroissant
df_outliers = df_outliers.sort_values('Nombre d\'outliers', ascending=False)

# Afficher le tableau complet
display(df_outliers)

# Créer le CSV nettoyé
df_clean = pd.concat(dfs_clean, ignore_index=True)
file_path_clean = f'../csv/STEP03/STEP03_maisons_dept{dept}_sans_outliers.csv'
df_clean.to_csv(file_path_clean, index=False, encoding='utf-8')
print(f"Fichier nettoyé sauvegardé : {file_path_clean}")

,Ville,Nombre de maisons,Nombre d'outliers,Moyenne avant outliers (€/m²),Moyenne après outliers (€/m²)
239,Vannes 56000,272,18,4429.34,4078.90
219,Sarzeau 56370,204,9,4663.96,4420.71
146,Ploemeur 56270,92,8,4499.87,4083.35
6,Auray 56400,77,7,3734.17,3621.09
89,Languidic 56440,88,7,2427.43,2454.96
7,Baden 56870,86,7,5086.58,4897.92
69,Guidel 56520,95,6,3844.56,3668.20
9,Baud 56150,90,6,3942.80,2060.98
94,Larmor-Baden 56870,33,6,6030.70,5771.63
0,Arradon 56610,44,5,5993.14,4970.85


Fichier nettoyé sauvegardé : ../csv/STEP03/STEP03_maisons_dept56_sans_outliers.csv
